# VQA Local Training — RTX 3060 12GB

**Cấu hình máy:** CPU Intel i7-14700K | RAM 64GB | GPU RTX 3060 12GB  
**Bài toán:** Visual Question Answering generative — LSTM Decoder sinh câu trả lời (VQA-E dataset)  
**4 Model so sánh:**

| Model | CNN | Attention | Pretrained |
|-------|-----|-----------|------------|
| **A** | SimpleCNN (scratch) | Không | Không |
| **B** | ResNet101 | Không | Có (ImageNet) |
| **C** | SimpleCNN Spatial (scratch) | Bahdanau | Không |
| **D** | ResNet101 Spatial | Bahdanau | Có (ImageNet) |

**Training strategy (3 phases):**
- Phase 1 (10 epochs): LR=1e-3, ResNet frozen (B/D), teacher forcing
- Phase 2 (5 epochs): LR=5e-4, unfreeze ResNet layer3+4 (B/D), differential LR
- Phase 3 (5 epochs): LR=2e-4, Scheduled Sampling (giảm exposure bias)

## 0. Environment Setup

In [24]:
import os
import sys
import torch

# ── Set working directory to project root ────────────────────────────────────
PROJECT_ROOT = "/home/mayxin/workspace/DeepLearning/new_vqa"
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

# ── PyTorch & CUDA info ───────────────────────────────────────────────────────
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability()
    vram_gb = gpu.total_memory / 1e9
    bf16_ok = cap[0] >= 8
    print(f"GPU             : {gpu.name}")
    print(f"VRAM            : {vram_gb:.1f} GB")
    print(f"Compute cap.    : {cap[0]}.{cap[1]}")
    print(f"BF16 support    : {'YES (Ampere+)' if bf16_ok else 'NO — will use FP16'}")
    # Enable cudnn auto-tune
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA not available, training will run on CPU (very slow)")

Working directory: /home/mayxin/workspace/DeepLearning/new_vqa

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3060
VRAM            : 12.5 GB
Compute cap.    : 8.6
BF16 support    : YES (Ampere+)


In [25]:
# ── Verify all required data paths exist ──────────────────────────────────────
required_paths = [
    ("data/raw/train2014",                      "COCO train2014 images"),
    ("data/raw/val2014",                        "COCO val2014 images"),
    ("data/vqa_e/VQA-E_train_set.json",         "VQA-E train annotations"),
    ("data/vqa_e/VQA-E_val_set.json",           "VQA-E val annotations"),
    ("data/processed/vocab_questions.json",     "Question vocabulary"),
    ("data/processed/vocab_answers.json",       "Answer vocabulary"),
    ("src/train.py",                            "Training script"),
    ("src/evaluate.py",                         "Evaluation script"),
    ("src/compare.py",                          "Comparison script"),
]

all_ok = True
for path, desc in required_paths:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ MISSING"
    print(f"{status}  {path:<50} ({desc})")
    if not exists:
        all_ok = False

if all_ok:
    import json
    vq = json.load(open("data/processed/vocab_questions.json"))
    va = json.load(open("data/processed/vocab_answers.json"))
    train_anns = json.load(open("data/vqa_e/VQA-E_train_set.json"))
    val_anns   = json.load(open("data/vqa_e/VQA-E_val_set.json"))
    print(f"\n=== Dataset Summary ===")
    print(f"Train samples   : {len(train_anns):,}")
    print(f"Val samples     : {len(val_anns):,}")
    print(f"Question vocab  : {len(vq['word2idx']):,} tokens")
    print(f"Answer vocab    : {len(va['word2idx']):,} tokens")
    print(f"\nSample entry    : {train_anns[0]}")
else:
    print("\n*** Fix missing paths before continuing! ***")

✓  data/raw/train2014                                 (COCO train2014 images)
✓  data/raw/val2014                                   (COCO val2014 images)
✓  data/vqa_e/VQA-E_train_set.json                    (VQA-E train annotations)
✓  data/vqa_e/VQA-E_val_set.json                      (VQA-E val annotations)
✓  data/processed/vocab_questions.json                (Question vocabulary)
✓  data/processed/vocab_answers.json                  (Answer vocabulary)
✓  src/train.py                                       (Training script)
✓  src/evaluate.py                                    (Evaluation script)
✓  src/compare.py                                     (Comparison script)

=== Dataset Summary ===
Train samples   : 181,298
Val samples     : 88,488
Question vocab  : 8,174 tokens
Answer vocab    : 3,706 tokens

Sample entry    : {'answer_type': 'other', 'explanation': ['Closeup of bins of food that include broccoli and bread.', 0.6791202865162379], 'question': 'What is the green stuff?',

## 1. Training Configuration (RTX 3060 12GB)

Batch sizes được chọn để chắc chắn không bị OOM trên VRAM 12GB với BF16 mixed precision:

| Model | batch_size | accum_steps | Effective batch |
|-------|-----------|-------------|----------------|
| A (SimpleCNN) | 64 | 2 | 128 |
| B (ResNet101) | 32 | 4 | 128 |
| C (SimpleCNN Spatial) | 32 | 2 | 64 |
| D (ResNet101 Spatial) | 16 | 4 | 64 |

> **Lưu ý:** Nếu bị OOM (out of memory), giảm `batch_size` thêm 1 nửa và tăng `accum_steps` gấp đôi.  
> Ví dụ: Model D bị OOM → dùng `--batch_size 8 --accum_steps 8`

In [26]:
# ── Config tập trung — chỉnh ở đây nếu cần ────────────────────────────────────
# Số workers cho DataLoader (i7-14700K có 20 cores)
NUM_WORKERS = 6

# Batch sizes cho RTX 3060 12GB (BF16 mixed precision)
BATCH = {'A': 64, 'B': 32, 'C': 32, 'D': 16}

# Gradient accumulation — effective_batch = BATCH × ACCUM
ACCUM = {'A': 2, 'B': 4, 'C': 2, 'D': 4}

# Early stopping patience (0 = disabled)
EARLY_STOP = 5

# Print effective batch sizes
print("=== Effective Batch Sizes ===")
for m in 'ABCD':
    print(f"  Model {m}: {BATCH[m]} × {ACCUM[m]} accum = {BATCH[m] * ACCUM[m]} effective")

# Helper: build common CLI args
def common_args(model, workers=NUM_WORKERS):
    return (
        f"--num_workers {workers} "
        f"--batch_size {BATCH[model]} "
        f"--accum_steps {ACCUM[model]} "
        f"--augment "
        f"--early_stopping {EARLY_STOP} "
        f"--weight_decay 1e-5"
    )

print("\n=== CLI args (Model A) ===")
print(common_args('A'))

=== Effective Batch Sizes ===
  Model A: 64 × 2 accum = 128 effective
  Model B: 32 × 4 accum = 128 effective
  Model C: 32 × 2 accum = 64 effective
  Model D: 16 × 4 accum = 64 effective

=== CLI args (Model A) ===
--num_workers 6 --batch_size 64 --accum_steps 2 --augment --early_stopping 5 --weight_decay 1e-5


## 1.5 (Optional) Sanity Test — 1 epoch, 1000 samples

Chạy cell này để kiểm tra pipeline hoạt động đúng trước khi train đầy đủ.  
Bỏ qua nếu bạn đã tự tin pipeline OK.

In [27]:
# Uncomment dòng bên dưới để chạy sanity test (khoảng 2-5 phút)
# !python src/train.py --model A --epochs 1 --batch_size 32 --num_workers 4
print("Sanity test cell — uncomment the line above to run")

Sanity test cell — uncomment the line above to run


## 2. Phase 1 — Baseline Training (10 epochs)

- LR = 1e-3 với LR warmup 3 epochs (lr/10 → lr)
- ResNet frozen hoàn toàn cho Model B/D
- Pure teacher forcing
- Checkpoint: `checkpoints/model_X_best.pth` và `checkpoints/model_X_epoch10.pth`

---
**Thời gian ước tính trên RTX 3060:** ~45-90 phút/model (tùy model)  
- Model A: ~45 phút  
- Model B: ~60 phút  
- Model C: ~75 phút  
- Model D: ~90 phút  

> Chạy tuần tự từng model. Có thể Resume nếu bị ngắt.

In [28]:
# ── Phase 1: Model A — SimpleCNN, no attention, scratch ──────────────────────
# Lightweight nhất, chạy nhanh nhất, là baseline reference
!python src/train.py \
    --model A \
    --epochs 10 \
    --lr 1e-3 \
    {common_args('A')}

Train: 181298 | Val: 88488
Weight decay     : 1.0e-05
LR Warmup        : 3 epochs (lr/10 → lr)
AMP: BFloat16 (Ampere+ detected — no GradScaler needed)
Model: A | Device: cuda
Gradient Accum   : 2 steps (effective batch = 128)
Data augmentation: ON (RandomHorizontalFlip + ColorJitter)
Early stopping   : patience=5
  0%|                                                    | 0/10 [00:00<?, ?it/s]Weight decay     : 1.0e-05
LR Warmup        : 3 epochs (lr/10 → lr)
AMP: BFloat16 (Ampere+ detected — no GradScaler needed)
Model: A | Device: cuda
Gradient Accum   : 2 steps (effective batch = 128)
Data augmentation: ON (RandomHorizontalFlip + ColorJitter)
Early stopping   : patience=5
  0%|                                                    | 0/10 [00:21<?, ?it/s]
Traceback (most recent call last):
  File "/home/mayxin/workspace/DeepLearning/new_vqa/src/train.py", line 546, in <module>
    train(model_type=args.model, epochs=args.epochs, lr=args.lr,
  File "/home/mayxin/workspace/DeepLearning/new

In [ ]:
# ── Phase 1: Model B — ResNet101 frozen, no attention, pretrained ─────────────
# ResNet frozen → chỉ train Q-encoder + Decoder + Fusion
!python src/train.py \
    --model B \
    --epochs 10 \
    --lr 1e-3 \
    {common_args('B')}

In [ ]:
# ── Phase 1: Model C — SimpleCNN Spatial, Bahdanau attention, scratch ─────────
# Dual attention (image regions + question) + Coverage mechanism
!python src/train.py \
    --model C \
    --epochs 10 \
    --lr 1e-3 \
    --coverage \
    {common_args('C')}

In [ ]:
# ── Phase 1: Model D — ResNet101 Spatial frozen, Bahdanau attention, pretrained
# Most complex: pretrained spatial features + dual attention + coverage
!python src/train.py \
    --model D \
    --epochs 10 \
    --lr 1e-3 \
    --coverage \
    {common_args('D')}

In [ ]:
# ── Kiểm tra checkpoint đã tạo ────────────────────────────────────────────────
import os
ckpt_dir = "checkpoints"
if os.path.exists(ckpt_dir):
    files = sorted(os.listdir(ckpt_dir))
    for f in files:
        fpath = os.path.join(ckpt_dir, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f:<45} {size_mb:6.1f} MB")
else:
    print("No checkpoints directory found — training may not have completed")

In [ ]:
# ── Compare sau Phase 1 (epoch 10) ────────────────────────────────────────────
!python src/compare.py --epoch 10 --num_samples 2000 --num_workers {NUM_WORKERS}

## 3. Phase 2 — Fine-tuning (5 epochs)

- Model A/C: tiếp tục train, LR=5e-4
- Model B/D: **unfreeze ResNet layer3+layer4**, differential LR (backbone LR × 0.1)
- Resume từ Phase 1 checkpoint

Checkpoint: `checkpoints/model_X_epoch15.pth`

In [ ]:
# ── Phase 2: Model A — continue training, LR=5e-4 ────────────────────────────
!python src/train.py \
    --model A \
    --epochs 5 \
    --lr 5e-4 \
    --resume checkpoints/model_a_resume.pth \
    {common_args('A')}

In [ ]:
# ── Phase 2: Model B — unfreeze ResNet layer3+4, differential LR ─────────────
# backbone LR = 5e-4 × 0.1 = 5e-5 | head LR = 5e-4
!python src/train.py \
    --model B \
    --epochs 5 \
    --lr 5e-4 \
    --finetune_cnn \
    --cnn_lr_factor 0.1 \
    --resume checkpoints/model_b_resume.pth \
    {common_args('B')}

In [ ]:
# ── Phase 2: Model C — continue training, LR=5e-4 ────────────────────────────
!python src/train.py \
    --model C \
    --epochs 5 \
    --lr 5e-4 \
    --coverage \
    --resume checkpoints/model_c_resume.pth \
    {common_args('C')}

In [ ]:
# ── Phase 2: Model D — unfreeze ResNet layer3+4, differential LR ─────────────
!python src/train.py \
    --model D \
    --epochs 5 \
    --lr 5e-4 \
    --coverage \
    --finetune_cnn \
    --cnn_lr_factor 0.1 \
    --resume checkpoints/model_d_resume.pth \
    {common_args('D')}

In [ ]:
# ── Compare sau Phase 2 (epoch 15) ────────────────────────────────────────────
!python src/compare.py --epoch 15 --num_samples 2000

## 4. Phase 3 — Scheduled Sampling (5 epochs)

- LR = 2e-4 cho tất cả models
- Scheduled Sampling: dần dần thay ground-truth token bằng model prediction
- `epsilon = k / (k + exp(epoch/k))`, k=5 → epsilon giảm từ ~0.5 xuống ~0.33
- Mục đích: giảm **exposure bias** (gap giữa train và inference)
- Checkpoint: `checkpoints/model_X_epoch20.pth`

In [ ]:
# ── Phase 3: Model A — Scheduled Sampling ────────────────────────────────────
!python src/train.py \
    --model A \
    --epochs 5 \
    --lr 2e-4 \
    --scheduled_sampling \
    --resume checkpoints/model_a_resume.pth \
    {common_args('A')}

In [ ]:
# ── Phase 3: Model B — Scheduled Sampling ────────────────────────────────────
!python src/train.py \
    --model B \
    --epochs 5 \
    --lr 2e-4 \
    --scheduled_sampling \
    --resume checkpoints/model_b_resume.pth \
    {common_args('B')}

In [ ]:
# ── Phase 3: Model C — Scheduled Sampling + Coverage ─────────────────────────
!python src/train.py \
    --model C \
    --epochs 5 \
    --lr 2e-4 \
    --scheduled_sampling \
    --coverage \
    --resume checkpoints/model_c_resume.pth \
    {common_args('C')}

In [ ]:
# ── Phase 3: Model D — Scheduled Sampling + Coverage ─────────────────────────
!python src/train.py \
    --model D \
    --epochs 5 \
    --lr 2e-4 \
    --scheduled_sampling \
    --coverage \
    --resume checkpoints/model_d_resume.pth \
    {common_args('D')}

## 5. Evaluation — Từng Model

Đánh giá chi tiết từng model với các metrics:
- **BLEU-4** ★ (primary)
- **METEOR** ★ (primary)
- BLEU-1/2/3 (reference)
- Exact Match (reference)

Có thể chạy sau từng phase bằng cách chỉnh `--checkpoint`

In [ ]:
# ── Evaluate Model A (greedy decode) ─────────────────────────────────────────
!python src/evaluate.py --model_type A --checkpoint checkpoints/model_a_best.pth

In [ ]:
# ── Evaluate Model B ──────────────────────────────────────────────────────────
!python src/evaluate.py --model_type B --checkpoint checkpoints/model_b_best.pth

In [ ]:
# ── Evaluate Model C ──────────────────────────────────────────────────────────
!python src/evaluate.py --model_type C --checkpoint checkpoints/model_c_best.pth

In [ ]:
# ── Evaluate Model D ──────────────────────────────────────────────────────────
!python src/evaluate.py --model_type D --checkpoint checkpoints/model_d_best.pth

In [ ]:
# ── Evaluate với Beam Search (width=5, n-gram blocking) ──────────────────────
# Cho kết quả tốt hơn greedy, nhưng chậm hơn ~5×
!python src/evaluate.py --model_type D --checkpoint checkpoints/model_d_best.pth \
    --beam_width 5 --no_repeat_ngram 3

## 6. Comparison Table — Tất cả 4 Models

So sánh 4 models ở 3 mốc: Phase 1 (epoch 10), Phase 2 (epoch 15), Phase 3 (epoch 20)

In [ ]:
# ── So sánh sau Phase 1 ───────────────────────────────────────────────────────
print("=" * 60)
print("PHASE 1 — Epoch 10")
print("=" * 60)
!python src/compare.py --epoch 10

In [ ]:
# ── So sánh sau Phase 2 ───────────────────────────────────────────────────────
print("=" * 60)
print("PHASE 2 — Epoch 15")
print("=" * 60)
!python src/compare.py --epoch 15

In [ ]:
# ── So sánh sau Phase 3 (Final) ───────────────────────────────────────────────
print("=" * 60)
print("PHASE 3 (FINAL) — Epoch 20")
print("=" * 60)
!python src/compare.py --epoch 20

In [ ]:
# ── So sánh với Beam Search (width=3) ────────────────────────────────────────
print("=" * 60)
print("FINAL — Beam Search width=3")
print("=" * 60)
!python src/compare.py --epoch 20 --beam_width 3 --no_repeat_ngram 3

## 7. Training Curves

In [ ]:
import os, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image as IPImage, display

# Generate plot using plot_curves.py
!python src/plot_curves.py --output checkpoints/training_curves.png

# Display in notebook
if os.path.exists("checkpoints/training_curves.png"):
    display(IPImage("checkpoints/training_curves.png"))
else:
    print("Plot not generated — train at least one model first")

In [ ]:
# ── Plot per-phase overlay (inline matplotlib) ────────────────────────────────
%matplotlib inline
import json, os
import matplotlib.pyplot as plt

MODEL_COLORS = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}
MODEL_LABELS = {
    'A': 'A: Scratch CNN, No Attn',
    'B': 'B: ResNet101, No Attn',
    'C': 'C: Scratch CNN, Attn',
    'D': 'D: ResNet101, Attn',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax_t, ax_v = axes

plotted_any = False
for m in 'ABCD':
    hp = f"checkpoints/history_model_{m.lower()}.json"
    if not os.path.exists(hp):
        continue
    h = json.load(open(hp))
    ep = list(range(1, len(h['train_loss']) + 1))
    ax_t.plot(ep, h['train_loss'], color=MODEL_COLORS[m], label=MODEL_LABELS[m], marker='o', ms=3)
    ax_v.plot(ep, h['val_loss'],   color=MODEL_COLORS[m], label=MODEL_LABELS[m], marker='s', ms=3, ls='--')
    # Mark phase boundaries
    for phase_ep in [10, 15]:
        if len(ep) >= phase_ep:
            ax_t.axvline(phase_ep, color='gray', ls=':', alpha=0.5)
            ax_v.axvline(phase_ep, color='gray', ls=':', alpha=0.5)
    plotted_any = True

if plotted_any:
    for ax, title in [(ax_t, 'Train Loss'), (ax_v, 'Validation Loss')]:
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title(title, fontsize=12)
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
    # Annotate phase boundaries
    for ax in [ax_t, ax_v]:
        ax.text(5,  ax.get_ylim()[1]*0.95, 'Phase 1', ha='center', fontsize=8, color='gray')
        ax.text(12.5, ax.get_ylim()[1]*0.95, 'Phase 2', ha='center', fontsize=8, color='gray')
        ax.text(17.5, ax.get_ylim()[1]*0.95, 'Phase 3', ha='center', fontsize=8, color='gray')
    fig.suptitle('VQA Training Curves — 4 Models (3 Phases)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig("checkpoints/training_curves_inline.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No history files found yet — train at least one model first")

## 8. Qualitative Analysis — Xem predictions mẫu

In [ ]:
import sys, os, json, torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

sys.path.insert(0, 'src')
from vocab import Vocabulary
from inference import get_model, greedy_decode, greedy_decode_with_attention

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load vocab
vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')

# Load a few validation samples
val_anns = json.load(open('data/vqa_e/VQA-E_val_set.json'))
val_img_dir = 'data/raw/val2014'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Select 5 random samples
import random
random.seed(42)
samples = random.sample(val_anns, 5)

print("Loaded validation samples")
for i, s in enumerate(samples):
    print(f"  {i+1}. Q: {s['question'][:60]}")
    print(f"     GT: {s.get('multiple_choice_answer', 'N/A')}")

In [ ]:
# ── Load trained models for comparison ───────────────────────────────────────
# Chỉ load model nếu checkpoint tồn tại
loaded_models = {}
for m in 'ABCD':
    ckpt = f'checkpoints/model_{m.lower()}_best.pth'
    if os.path.exists(ckpt):
        model = get_model(m, len(vocab_q), len(vocab_a))
        model.load_state_dict(torch.load(ckpt, map_location='cpu'))
        model.to(DEVICE).eval()
        loaded_models[m] = model
        print(f"  Loaded Model {m} from {ckpt}")
    else:
        print(f"  [SKIP] Model {m}: {ckpt} not found")

print(f"\nLoaded {len(loaded_models)} models: {list(loaded_models.keys())}")

In [ ]:
# ── Show predictions side by side ────────────────────────────────────────────
%matplotlib inline

n_samples = min(3, len(samples))
n_models  = len(loaded_models)

if n_models == 0:
    print("No models loaded — train at least one model first")
else:
    for idx in range(n_samples):
        ann    = samples[idx]
        img_id = ann['img_id']
        q_text = ann['question']
        gt_ans = ann.get('multiple_choice_answer', '')
        gt_exp = ann.get('explanation', [''])[0] if ann.get('explanation') else ''
        gt_full = f"{gt_ans} because {gt_exp}" if gt_exp else gt_ans

        img_path = os.path.join(val_img_dir, f"COCO_val2014_{img_id:012d}.jpg")
        if not os.path.exists(img_path):
            print(f"Image not found: {img_path}")
            continue

        img_pil    = Image.open(img_path).convert("RGB")
        img_tensor = transform(img_pil)
        q_tensor   = torch.tensor(vocab_q.numericalize(q_text), dtype=torch.long)

        # Get predictions from all models
        preds = {}
        for m, model in loaded_models.items():
            if m in ('A', 'B'):
                pred = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
            else:
                pred = greedy_decode_with_attention(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
            preds[m] = pred

        # Display
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        ax.imshow(img_pil)
        ax.axis('off')
        ax.set_title(f"Q: {q_text}", fontsize=9, wrap=True)
        plt.tight_layout()
        plt.show()

        print(f"  Ground Truth : {gt_full}")
        for m, pred in preds.items():
            print(f"  Model {m}      : {pred}")
        print("-" * 60)

## 9. Attention Visualization (Model C/D)

Visualize vùng ảnh mà model chú ý khi sinh từng token.

In [ ]:
# ── Attention Heatmap cho Model C hoặc D ─────────────────────────────────────
# Cần model C hoặc D đã được train
attn_model_type = 'D' if 'D' in loaded_models else ('C' if 'C' in loaded_models else None)

if attn_model_type is None:
    print("Model C/D chưa được train. Hãy train Model C hoặc D trước.")
else:
    print(f"Using Model {attn_model_type} for attention visualization")
    # visualize.py handles this — run as script with an example image
    sample = samples[0]
    img_id = sample['img_id']
    q_text = sample['question']
    img_path = f"data/raw/val2014/COCO_val2014_{img_id:012d}.jpg"
    ckpt = f"checkpoints/model_{attn_model_type.lower()}_best.pth"
    if os.path.exists(img_path) and os.path.exists(ckpt):
        !python src/visualize.py \
            --model_type {attn_model_type} \
            --checkpoint {ckpt} \
            --image_path "{img_path}" \
            --question "{q_text}" \
            --output checkpoints/attention_viz.png
        if os.path.exists('checkpoints/attention_viz.png'):
            from IPython.display import Image as IPImage, display
            display(IPImage('checkpoints/attention_viz.png'))
    else:
        print(f"Missing: {img_path} or {ckpt}")

## 10. Resume Training (nếu bị ngắt)

Nếu training bị ngắt giữa chừng, dùng `--resume` để tiếp tục từ checkpoint cuối.  
File `checkpoints/model_X_resume.pth` được overwrite sau mỗi epoch.

```bash
# Resume Model A Phase 1 (ví dụ bị dừng ở epoch 6/10)
python src/train.py --model A --epochs 10 --lr 1e-3 \
    --resume checkpoints/model_a_resume.pth \
    --batch_size 64 --accum_steps 2 --augment --num_workers 6
```

> **Note:** `--epochs 10` là tổng số epochs tính từ đầu. Nếu đã chạy 6 epochs,  
> sẽ tiếp tục từ epoch 7 cho đến epoch 10 (4 epochs còn lại).

In [ ]:
# ── Kiểm tra trạng thái training hiện tại ────────────────────────────────────
import json, os

print("=== Training Progress ===")
for m in 'ABCD':
    history_path = f"checkpoints/history_model_{m.lower()}.json"
    resume_path  = f"checkpoints/model_{m.lower()}_resume.pth"
    best_path    = f"checkpoints/model_{m.lower()}_best.pth"

    if os.path.exists(history_path):
        h = json.load(open(history_path))
        n_epochs = len(h['train_loss'])
        best_val = min(h['val_loss']) if h['val_loss'] else float('inf')
        last_val = h['val_loss'][-1] if h['val_loss'] else float('inf')
        phase = 'Phase 1' if n_epochs <= 10 else ('Phase 2' if n_epochs <= 15 else 'Phase 3')
        has_ckpt = '✓' if os.path.exists(resume_path) else '✗'
        has_best = '✓' if os.path.exists(best_path) else '✗'
        print(f"  Model {m}: {n_epochs:2d} epochs | {phase} | best_val={best_val:.4f} | last_val={last_val:.4f} | resume={has_ckpt} | best={has_best}")
    else:
        print(f"  Model {m}: not started yet")

---

## Appendix: Useful Commands

```bash
# Xem GPU usage trong lúc train (terminal riêng)
watch -n 1 nvidia-smi

# Evaluate nhanh với 500 samples
python src/evaluate.py --model_type A --num_samples 500

# So sánh với beam search
python src/compare.py --epoch 10 --beam_width 3 --num_samples 1000

# Plot training curves
python src/plot_curves.py

# Rebuild vocab (nếu cần)
python src/scripts/1_build_vocab.py
```

### Expected Results (tham khảo)

| Model | BLEU-4 | METEOR | Notes |
|-------|--------|--------|-------|
| A | 0.05–0.10 | 0.10–0.18 | Baseline |
| B | 0.07–0.12 | 0.12–0.20 | Pretrained CNN |
| C | 0.06–0.11 | 0.11–0.19 | Attention helps long answers |
| D | 0.08–0.15 | 0.13–0.22 | Best expected |

> Kết quả thực tế phụ thuộc vào số epochs, learning rate, và dataset size.